# Proof of Work — Analysis Notebook

Computes the 3 pre-registered claims (C1 cache reuse, C3 rework reduction, C6 throughput) from `data/joined.parquet` and stamps headline numbers + 6 charts.

> **Risk #7 (Plan 38).** This notebook is authored to run on **placeholder fixtures only** until Plan 38 is in `Plans/Archive/`. Running on real data before plan-merge breaks the pre-registration commitment. Set `BONSAI_EVAL_USE_FIXTURES=1` (default) to run on `tests/fixtures/joined.parquet`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

USE_FIXTURES = os.environ.get("BONSAI_EVAL_USE_FIXTURES", "1") != "0"
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if USE_FIXTURES:
    JOINED = REPO_ROOT / "tests" / "fixtures" / "joined.parquet"
else:
    JOINED = REPO_ROOT / "data" / "joined.parquet"

CHARTS_DIR = REPO_ROOT / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loading {JOINED} (fixtures={USE_FIXTURES})")
df = pd.read_parquet(JOINED)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df.head()

## Cut-over

2026-04-14 = `4dfd3f4` (station/ adopted Bonsai).

In [ ]:
CUTOVER = pd.Timestamp("2026-04-14")
df["period"] = df["date"].apply(lambda d: "post" if d >= CUTOVER else "pre")

## C1 — Cache reuse

`cache_hit_rate = cache_read_tokens / (cache_read_tokens + input_tokens)`.

In [ ]:
denom = (df["cache_read_tokens"] + df["input_tokens"]).astype(float).replace(0.0, float("nan"))
df["cache_hit_rate"] = df["cache_read_tokens"].astype(float) / denom

fig, ax = plt.subplots(figsize=(10, 4))
for slug, sub in df.groupby("project_slug"):
    ax.plot(sub["date"], sub["cache_hit_rate"], label=slug, marker="o", linewidth=1)
ax.axvline(CUTOVER, color="red", linestyle="--", label="cutover")
ax.set_title("C1 — cache hit rate (placeholder data)")
ax.set_ylabel("cache_read / (cache_read + input)")
ax.legend()
fig.tight_layout()
fig.savefig(CHARTS_DIR / "c1-cache.png", dpi=120)
plt.close(fig)
print("saved charts/c1-cache.png")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    df.groupby("period")["cache_hit_rate"].median().index,
    df.groupby("period")["cache_hit_rate"].median().values,
)
ax.set_title("C1 — median cache hit rate, pre vs post (placeholder data)")
fig.tight_layout()
fig.savefig(CHARTS_DIR / "c1-memory.png", dpi=120)
plt.close(fig)
print("saved charts/c1-memory.png")

## C3 — Rework reduction

`rework_ratio = rework_commits / feat_commits` per 14d rolling window.

In [ ]:
rework = df.groupby("date")[["rework_commits", "feat_commits"]].sum().reset_index()
rework_denom = rework["feat_commits"].astype(float).replace(0.0, float("nan"))
rework["rework_ratio"] = rework["rework_commits"].astype(float) / rework_denom

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rework["date"], rework["rework_ratio"], marker="o", linewidth=1)
ax.axvline(CUTOVER, color="red", linestyle="--", label="cutover")
ax.set_title("C3 — rework ratio (placeholder data)")
ax.set_ylabel("fix(scope) within 24h of feat(scope) / feat")
ax.legend()
fig.tight_layout()
fig.savefig(CHARTS_DIR / "c3-rework.png", dpi=120)
plt.close(fig)
print("saved charts/c3-rework.png")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(rework["date"], rework["rework_commits"], label="rework_commits")
ax.axvline(CUTOVER, color="red", linestyle="--", label="cutover")
ax.set_title("C3 — rework commits per day (placeholder data)")
ax.legend()
fig.tight_layout()
fig.savefig(CHARTS_DIR / "c3-revert.png", dpi=120)
plt.close(fig)
print("saved charts/c3-revert.png")

## C6 — Throughput

In [ ]:
throughput = df.groupby("date")["prs_merged"].sum().reset_index()
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(throughput["date"], throughput["prs_merged"])
ax.axvline(CUTOVER, color="red", linestyle="--", label="cutover")
ax.set_title("C6 — PRs merged per day (placeholder data)")
ax.legend()
fig.tight_layout()
fig.savefig(CHARTS_DIR / "c6-plans.png", dpi=120)
plt.close(fig)
print("saved charts/c6-plans.png")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df["date"], df["tool_calls"], marker="o", linewidth=1)
ax.axvline(CUTOVER, color="red", linestyle="--", label="cutover")
ax.set_title("C6 — tool-call volume per day (placeholder proxy for activity)")
ax.legend()
fig.tight_layout()
fig.savefig(CHARTS_DIR / "c6-latency.png", dpi=120)
plt.close(fig)
print("saved charts/c6-latency.png")

## Summary

Six chart PNGs under `charts/`, plus printed-to-stdout summary stats. Pipe these into `PROOF-OF-WORK.md` once data is real (Plan 38 must be archived first).